# Vietnamese Medical NER / Assertion / Mapping — Kaggle runner

Chạy pipeline self-host (Qwen3-8B qua Ollama + ViHealthBERT/e5) trên 100 file `.txt`.

**Trước khi chạy — bật trong panel Settings bên phải:**
1. **Accelerator = GPU T4 x2** (hoặc P100)
2. **Internet = On** (bắt buộc: tải Ollama/model + gọi RxNorm API)
3. **Add Data** → chọn Kaggle Dataset chứa `input/` (100 .txt) + `ICD10_master_active.xlsx`

Chạy tuần tự từng cell. Khi chạy full 100 file, dùng **Save & Run All (Commit)** để job chạy độc lập.

## 1. Cấu hình — chỉnh cho khớp Dataset của bạn

In [ ]:
# ==== CHỈNH slug cho khớp Kaggle Dataset đã Add Data ====
DATASET_SLUG = "my-icd-med-data"   # đổi thành tên dataset của bạn

INPUT_DIR  = f"/kaggle/input/{DATASET_SLUG}/input"
ICD10_PATH = f"/kaggle/input/{DATASET_SLUG}/ICD10_master_active.xlsx"

REPO_URL = "https://github.com/jasmine95dn/vn-medical-ner-assertion.git"
REPO_DIR = "/kaggle/working/vn-medical-ner-assertion"

import os
print("INPUT_DIR tồn tại:", os.path.isdir(INPUT_DIR))
print("ICD10_PATH tồn tại:", os.path.isfile(ICD10_PATH))
assert os.path.isdir(INPUT_DIR),  "Sai INPUT_DIR — kiểm tra lại DATASET_SLUG và Add Data"
assert os.path.isfile(ICD10_PATH), "Sai ICD10_PATH — kiểm tra tên file .xlsx trong dataset"

## 2. Lấy code từ GitHub (clone lần đầu, các lần sau tự `git pull`)

In [ ]:
import os, subprocess
if os.path.isdir(REPO_DIR):
    print("repo đã có → git pull")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

## 3. Cài thư viện Python

In [ ]:
!pip install -q -r requirements.txt --break-system-packages

## 4. Cài & khởi động Ollama, tải model Qwen3-8B

Server chạy nền; lần đầu `ollama pull qwen3:8b` tải ~5GB nên hơi lâu.

In [ ]:
import subprocess, time, os

# cài Ollama nếu chưa có
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)

# chạy server nền
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
subprocess.Popen(["ollama", "serve"])
time.sleep(8)  # chờ server sẵn sàng

subprocess.run(["ollama", "pull", "qwen3:8b"], check=True)
print("Ollama sẵn sàng.")

## 5. Smoke test — 3 file, bỏ candidate (nhanh, kiểm pipeline chạy được)

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}" --limit 3 --no-candidates

In [ ]:
# xem thử vài entity đầu ra
import json, glob
for f in sorted(glob.glob("output/*.json"))[:3]:
    print("===", f, "===")
    data = json.load(open(f, encoding="utf-8"))
    print(json.dumps(data[:5], ensure_ascii=False, indent=2))

## 6. Chấm điểm nhanh trên validation set (NER + assertion)

In [ ]:
!python evaluate.py --run --save-pred output/val_pred.json

## 7. Chạy FULL 100 file (có candidate mapping ICD-10 + RxNorm)

Lần đầu sẽ tải 2 model embedding (ViHealthBERT + e5-base) từ HuggingFace.
Nên chạy bằng **Save & Run All (Commit)** để không cần giữ tab mở.

In [ ]:
!python main.py --input-dir "{INPUT_DIR}" --icd10-path "{ICD10_PATH}"

## 8. Đóng gói output để tải về

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/output", "zip", "output")
print("Đã nén →/kaggle/working/output.zip (tải ở tab Output)")
import glob
print("Số file JSON:", len(glob.glob("output/*.json")))